<h1>Judul Karya : <b>QRGuards</b></h1>

## **Project Overview**

QRGuards adalah sistem berbasis **Artificial Intelligence** yang digunakan untuk mendeteksi apakah URL hasil pemindaian QR Code mengarah ke situs **phishing** atau situs **legitimate**. Sistem ini dirancang untuk membantu pengguna melakukan analisis keamanan sebelum membuka tautan dari QR Code.

QR Code banyak digunakan pada pembayaran digital, absensi, promosi, formulir online, dan akses layanan web. Namun, QR Code juga dapat disalahgunakan untuk menyembunyikan URL berbahaya karena pengguna tidak dapat melihat alamat tujuan secara langsung sebelum melakukan pemindaian. Hal ini membuat serangan phishing berbasis QR Code menjadi sulit dikenali secara manual.

Metode deteksi konvensional yang hanya melihat daftar blacklist URL memiliki keterbatasan karena phishing site dapat berubah dengan cepat. Oleh karena itu, proyek ini menggunakan pendekatan **Hybrid Deep Learning** dengan menggabungkan pemahaman semantik URL melalui **BERT embedding** dan analisis struktur URL melalui **manual feature engineering**.

Dalam konteks deteksi phishing URL, sistem menghadapi beberapa tantangan utama:

* **URL Obfuscation**: URL phishing sering menggunakan karakter khusus, angka, simbol, subdomain panjang, atau path yang dibuat menyerupai situs resmi.
* **Variasi Pola Phishing**: URL phishing dapat memiliki struktur yang sangat beragam sehingga sulit dideteksi hanya menggunakan aturan manual.
* **Ketidakseimbangan Kelas**: Dataset memiliki distribusi kelas legitimate dan phishing yang tidak sepenuhnya seimbang.
* **Generalization ke Data Eksternal**: Model tidak hanya perlu bagus pada validation set, tetapi juga harus tetap bekerja pada dataset eksternal yang tidak digunakan saat training.
* **Kombinasi Fitur Semantik dan Struktural**: Sistem perlu memahami makna/pola URL sekaligus karakteristik teknis URL seperti panjang, jumlah simbol, query, fragment, dan entropy.

---

## **Problem Statement**

Proyek ini berfokus pada pengembangan model **binary classification** untuk mendeteksi apakah suatu URL termasuk kategori **legitimate** atau **phishing**. Target utama yang digunakan adalah kolom `type`, yang dipetakan menjadi dua kelas:

* `0` → Legitimate / Benign / Good
* `1` → Phishing / Malicious / Bad

Model dibangun menggunakan pendekatan hybrid, yaitu menggabungkan:

1. **BERT URL Embedding**  
   URL dikonversi menjadi representasi vektor berdimensi 768 menggunakan model `distilbert-base-nli-mean-tokens`. Embedding ini digunakan untuk menangkap pola semantik dan representasi tekstual dari URL.

2. **Manual URL Feature Engineering**  
   URL diekstraksi menjadi fitur numerik berbasis struktur URL, seperti panjang URL, jumlah digit, jumlah karakter khusus, jumlah slash, query, fragment, anchor, dan entropy.

3. **BiLSTM with Attention**  
   Embedding BERT diproses menggunakan BiLSTM untuk menangkap representasi urutan dari URL, lalu attention layer digunakan untuk memberi bobot pada representasi yang lebih penting.

4. **Feature Fusion**  
   Output dari BiLSTM-Attention digabungkan dengan manual URL features sebelum masuk ke classifier akhir.

Tantangan utama dalam proyek ini meliputi:

* **URL Feature Extraction**: Mengekstrak fitur penting dari URL secara konsisten meskipun format URL berbeda-beda.
* **Semantic Representation**: Menggunakan BERT embedding agar model dapat menangkap pola URL yang tidak hanya berbasis karakter manual.
* **External Test Evaluation**: Menguji model pada dataset eksternal untuk melihat kemampuan generalisasi.
* **Class Imbalance Handling**: Mengevaluasi model menggunakan metrik seperti precision, recall, F1-score, macro average, dan confusion matrix.
* **Data Leakage Checking**: Mengecek overlap antara dataset training dan dataset testing agar evaluasi lebih valid.

---

## **Expected Impact**

Solusi QRGuards diharapkan dapat membantu pengguna mendeteksi potensi phishing dari URL yang tersembunyi di balik QR Code sebelum tautan tersebut dibuka.

Dampak yang diharapkan dari proyek ini adalah:

* **Early Phishing Detection**  
  Membantu mendeteksi URL phishing lebih awal sebelum pengguna membuka link berbahaya.

* **Improved QR Code Security**  
  Memberikan lapisan keamanan tambahan pada proses pemindaian QR Code.

* **Data-Driven URL Risk Analysis**  
  Mengubah proses pengecekan URL dari manual menjadi berbasis model prediktif.

* **Better Generalization**  
  Menguji model pada dataset eksternal agar sistem tidak hanya bekerja pada data training, tetapi juga pada data baru.

* **User Protection**  
  Mengurangi risiko pengguna terkena pencurian data, credential theft, fake login page, atau social engineering melalui QR phishing.

* **Decision Support System**  
  Sistem dapat memberikan hasil prediksi berupa legitimate/phishing sebagai dasar peringatan keamanan kepada pengguna.

---

## **Success Metrics**

Keberhasilan model diukur berdasarkan performa klasifikasi terhadap target `type`.

Metrik evaluasi utama yang digunakan adalah:

* **Accuracy**  
  Mengukur persentase prediksi benar dari seluruh data.

* **Precision**  
  Mengukur seberapa tepat model ketika memprediksi URL sebagai phishing.

* **Recall**  
  Mengukur kemampuan model dalam mendeteksi seluruh URL phishing yang benar-benar phishing.

* **F1-Score**  
  Digunakan sebagai metrik utama karena menyeimbangkan precision dan recall.

* **Macro Average**  
  Digunakan untuk melihat performa rata-rata antar kelas tanpa terlalu dipengaruhi dominasi kelas mayoritas.

* **Confusion Matrix**  
  Digunakan untuk melihat jumlah prediksi benar dan salah pada masing-masing kelas legitimate dan phishing.

Berdasarkan hasil notebook, model memperoleh performa pada **external test set** sebagai berikut:

| Metric | Score |
|---|---:|
| Accuracy | 87.72% |
| Precision | 77.99% |
| Recall | 85.06% |
| F1-Score | 81.37% |
| Macro F1-Score | 86.00% |
| Weighted F1-Score | 88.00% |

Confusion matrix pada external test set:

| Actual / Predicted | Legitimate | Phishing |
|---|---:|---:|
| Legitimate | 307,518 | 38,220 |
| Phishing | 23,789 | 135,455 |

Secara keseluruhan, proyek dianggap berhasil apabila model mampu mendeteksi phishing URL dengan nilai **recall dan F1-score yang baik**, karena dalam kasus keamanan siber, gagal mendeteksi URL phishing dapat berdampak lebih serius dibandingkan false alarm.

---

## Pipeline Keseluruhan Tahap Pemodelan

```text
┌──────────────────────────────────────────────┐
│                 Raw Dataset                  │
│                                              │
│ Dataset utama:                               │
│ - phishing_site_urls.csv                     │
│                                              │
│ Dataset eksternal:                           │
│ - Phishing URLs.csv                          │
│ - URL dataset.csv                            │
│                                              │
│ Columns used:                                │
│ - url                                        │
│ - type                                       │
└───────────────────────┬──────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────┐
│              Label Processing                 │
│                                              │
│ Label mapping:                               │
│ - benign / legitimate / good → 0             │
│ - phishing / malicious / bad → 1             │
│                                              │
│ Target:                                      │
│ - type                                       │
│                                              │
│ Classes:                                     │
│ - 0 = Legitimate                             │
│ - 1 = Phishing                               │
└───────────────────────┬──────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────┐
│              Data Validation                  │
│                                              │
│ - Check missing value                        │
│ - Parse URL using urlparse                   │
│ - Remove invalid URL                         │
│ - Check overlap between train and test       │
│                                              │
│ Output:                                      │
│ - Clean train dataset                        │
│ - Clean external test dataset                │
└───────────────────────┬──────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────┐
│              Data Leakage Check               │
│                                              │
│ Overlap result:                              │
│ - Train vs Phishing URLs.csv = 0             │
│ - Train vs URL dataset.csv = 37              │
│                                              │
│ Note:                                        │
│ 37 overlap rows should be removed to reduce  │
│ leakage risk in final experiment.            │
└───────────────────────┬──────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────┐
│           Manual URL Feature Engineering      │
│                                              │
│ Extracted features:                          │
│ - url_length                                 │
│ - number_of_dots_in_url                      │
│ - having_repeated_digits_in_url              │
│ - number_of_digits_in_url                    │
│ - number_of_special_char_in_url              │
│ - number_of_hyphens_in_url                   │
│ - number_of_underline_in_url                 │
│ - number_of_slash_in_url                     │
│ - number_of_questionmark_in_url              │
│ - number_of_equal_in_url                     │
│ - number_of_at_in_url                        │
│ - number_of_dollar_in_url                    │
│ - number_of_exclamation_in_url               │
│ - number_of_hashtag_in_url                   │
│ - number_of_percent_in_url                   │
│ - path_length                                │
│ - having_query                               │
│ - having_fragment                            │
│ - having_anchor                              │
│ - entropy_of_url                             │
└───────────────────────┬──────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────┐
│              BERT Embedding                   │
│                                              │
│ Model:                                       │
│ - distilbert-base-nli-mean-tokens            │
│                                              │
│ Process:                                     │
│ - Convert URL text into 768-dim embedding    │
│ - Store embedding using numpy memmap         │
│ - Align embedding with extracted features    │
│                                              │
│ Output:                                      │
│ - train_bert_embeddings_aligned              │
│ - test_bert_embeddings_aligned               │
└───────────────────────┬──────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────┐
│              Train-Val Split                  │
│                                              │
│ Method:                                      │
│ - train_test_split                           │
│ - test_size = 0.15                           │
│ - stratify = y                               │
│ - random_state = 42                          │
│                                              │
│ Output:                                      │
│ - Training data                              │
│ - Validation data                            │
└───────────────────────┬──────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────┐
│              Feature Scaling                  │
│                                              │
│ Manual features are scaled using:            │
│ - StandardScaler                             │
│                                              │
│ Scaling fit:                                 │
│ - Fit only on training data                  │
│ - Transform validation and test data         │
└───────────────────────┬──────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────┐
│              Dataset Loader                   │
│                                              │
│ Custom PyTorch Dataset:                      │
│ - BERT embedding                             │
│ - Manual URL features                        │
│ - Binary label                               │
│                                              │
│ DataLoader:                                  │
│ - batch_size = 16                            │
│ - shuffle = True for train                   │
│ - shuffle = False for validation/test        │
└───────────────────────┬──────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────┐
│              Hybrid Model                     │
│          BiLSTM-Attention + Manual Features   │
│                                              │
│ BERT branch:                                 │
│ - Input: 768-dim embedding                   │
│ - BiLSTM hidden size = 256                   │
│ - Bidirectional = True                       │
│ - Attention layer                            │
│                                              │
│ Manual feature branch:                       │
│ - Input: 20 manual features                  │
│ - Dense layer: 20 → 64                       │
│ - ReLU                                       │
│ - Dropout = 0.1                              │
│                                              │
│ Fusion:                                      │
│ - Concatenate BERT context + manual output   │
│ - Final classifier with Sigmoid              │
│                                              │
│ Output:                                      │
│ - Probability of phishing                    │
└───────────────────────┬──────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────┐
│              Model Training                   │
│                                              │
│ Loss function:                               │
│ - Binary Cross Entropy Loss                  │
│                                              │
│ Optimizer:                                   │
│ - AdamW                                      │
│ - learning_rate = 2e-5                       │
│                                              │
│ Epoch:                                       │
│ - 50                                         │
│                                              │
│ Checkpoint:                                  │
│ - Save best model based on validation loss   │
└───────────────────────┬──────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────┐
│              Model Evaluation                 │
│                                              │
│ Evaluation data:                             │
│ - External test dataset                      │
│                                              │
│ Metrics:                                     │
│ - Accuracy                                   │
│ - Precision                                  │
│ - Recall                                     │
│ - F1-score                                   │
│ - Classification report                      │
│ - Confusion matrix                           │
└───────────────────────┬──────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────┐
│                Final Output                   │
│                                              │
│ - Predicted class: Legitimate / Phishing     │
│ - Phishing probability                       │
│ - Model evaluation result                    │
│ - Saved model checkpoint                     │
│ - QR URL security decision support           │
└──────────────────────────────────────────────┘

# SETUP

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import re
import tldextract
from urllib.parse import urlparse
import torch.nn as nn
from tqdm.auto import tqdm
import numpy as np
import torch
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from IPython.display import FileLink

tqdm.pandas()
pd.set_option('display.max_columns', None)
%matplotlib inline

# Data Processing

In [12]:
SPECIAL_CHARS = set(['"', '#', '$', '%', '&', '~'])

def add_scheme_if_missing(url):
    url = str(url).strip()
    if not re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", url):
        url = "http://" + url
    return url

def extract_domain_subdomain(url):
    ext = tldextract.extract(url)

    if ext.suffix:
        main_domain = ext.domain + "." + ext.suffix
    else:
        main_domain = ext.domain

    subdomain = ext.subdomain

    if subdomain:
        subdomain_parts = subdomain.split(".")
    else:
        subdomain_parts = []

    return main_domain, subdomain, subdomain_parts

def get_parsed_url(url):
    try:
        url_with_scheme = add_scheme_if_missing(url)
        parsed = urlparse(url_with_scheme)
        return parsed
    except ValueError:
        return None




def count_digits(text):
    return sum(c.isdigit() for c in str(text))


def has_digits(text):
    return int(count_digits(text) > 0)


def has_repeated_digits(text):
    text = str(text)
    return int(bool(re.search(r"(\d)\1+", text)))



def count_special_url(text):
    return sum(c in SPECIAL_CHARS for c in str(text))

def count_special_domain(text):
    return sum(c in SPECIAL_CHARS for c in str(text))


def shannon_entropy(text):
    text = str(text)
    if len(text) == 0:
        return 0.0

    counter = Counter(text)
    length = len(text)

    entropy = 0.0
    for count in counter.values():
        p = count / length
        entropy -= p * math.log2(p)

    return entropy



def extract_features(url):
    original_url = str(url).strip()

    parsed = get_parsed_url(original_url)
    if parsed is None:
        return None
    path = parsed.path
    query = parsed.query
    fragment = parsed.fragment

    domain, subdomain, subdomain_parts = extract_domain_subdomain(original_url)
   

    features = {}

    features["url_length"] = len(original_url)
    features["number_of_dots_in_url"] = original_url.count(".")
    features["having_repeated_digits_in_url"] = has_repeated_digits(original_url)
    features["number_of_digits_in_url"] = count_digits(original_url)
    features["number_of_special_char_in_url"] = count_special_url(original_url)
    features["number_of_hyphens_in_url"] = original_url.count("-")
    features["number_of_underline_in_url"] = original_url.count("_")
    features["number_of_slash_in_url"] = original_url.count("/") + original_url.count("\\")
    features["number_of_questionmark_in_url"] = original_url.count("?")
    features["number_of_equal_in_url"] = original_url.count("=")
    features["number_of_at_in_url"] = original_url.count("@")
    features["number_of_dollar_in_url"] = original_url.count("$")
    features["number_of_exclamation_in_url"] = original_url.count("!")
    features["number_of_hashtag_in_url"] = original_url.count("#")
    features["number_of_percent_in_url"] = original_url.count("%")


    features["path_length"] = len(path)
    features["having_query"] = int(len(query) > 0)
    features["having_fragment"] = int(len(fragment) > 0)
    features["having_anchor"] = features["having_fragment"] 

    features["entropy_of_url"] = shannon_entropy(original_url)
    return features




In [13]:
df = pd.read_csv("/kaggle/input/datasets/rasyidbomantoro/phising-urls/phishing_site_urls.csv")
test = pd.read_csv("/kaggle/input/datasets/rasyidbomantoro/phising-urls/Phishing URLs.csv")
test_2 = pd.read_csv("/kaggle/input/datasets/rasyidbomantoro/phising-urls/URL dataset.csv")

df = df.rename(columns={
    "URL": "url",
    "Url": "url",
    "Type": "type"
})

test = test.rename(columns={
    "URL": "url",
    "Url": "url",
    "Type": "type"
})

test_2 = test_2.rename(columns={
    "URL": "url",
    "Url": "url",
    "Type": "type"
})
test_all = pd.concat([test, test_2], axis=0).reset_index(drop=True)

print("Before mapping test_all:")
print(test_all["type"].value_counts(dropna=False))
print("Before mapping:")
print(df["type"].value_counts(dropna=False))
print(test["type"].value_counts(dropna=False))
print(test_2["type"].value_counts(dropna=False))

Before mapping test_all:
type
legitimate    345738
phishing      104438
Phishing       54807
Name: count, dtype: int64
Before mapping:
type
good    392924
bad     156422
Name: count, dtype: int64
type
Phishing    54807
Name: count, dtype: int64
type
legitimate    345738
phishing      104438
Name: count, dtype: int64


In [14]:
label_map = {
    "benign": 0,
    "Benign": 0,
    "legitimate": 0,
    "Legitimate": 0,
    "good": 0,
    "Good": 0,

    "bad": 1,
    "Bad": 1,
    "phishing": 1,
    "Phishing": 1,
    "malicious": 1,
    "Malicious": 1
}

df["type"] = df["type"].map(label_map)
test_all["type"] = test_all["type"].map(label_map)
test_all["type"].value_counts()

type
0    345738
1    159245
Name: count, dtype: int64

In [15]:
df.isna().sum(), test.isna().sum(),

(url     0
 type    0
 dtype: int64,
 url     0
 type    0
 dtype: int64)

In [30]:
bad_rows = []

for idx, url in tqdm(test_all["url"].astype(str).items(), total=len(test_all)):
    try:
        _ = urlparse(url)
    except ValueError as e:
        bad_rows.append((idx, url, str(e)))

bad_idx = [row[0] for row in bad_rows]

test_all = test_all.drop(index=bad_idx).reset_index(drop=True)

print("Before:", test_all.shape)
print("After :", test_all.shape)
print(test_all["type"].value_counts())

100%|██████████| 504982/504982 [00:02<00:00, 198408.87it/s]

Before: (504982, 2)
After : (504982, 2)
type
0    345738
1    159244
Name: count, dtype: int64


In [31]:
df_urls = df["url"].astype(str).str.strip().str.lower()
test_urls = test["url"].astype(str).str.strip().str.lower()
test2_urls = test_2["url"].astype(str).str.strip().str.lower()

overlap_df_test = set(df_urls) & set(test_urls)
overlap_df_test2 = set(df_urls) & set(test2_urls)

print("Overlap df vs test:", len(overlap_df_test))
print("Overlap df vs test_2:", len(overlap_df_test2))

Overlap df vs test: 0
Overlap df vs test_2: 37


# Embedding Generation

In [ ]:
bert_model = SentenceTransformer("distilbert-base-nli-mean-tokens")
device = "cuda" if torch.cuda.is_available() else "cpu"
bert_model = bert_model.to(device)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [33]:
def make_bert_embeddings(data, file_name, bert_model, device, url_col="url", batch_size=32):
    urls = data[url_col].astype(str).tolist()

    sample_embedding = bert_model.encode([urls[0]], device=device)
    embedding_dim = sample_embedding.shape[1]

    embeddings = np.memmap(
        file_name,
        dtype="float32",
        mode="w+",
        shape=(len(urls), embedding_dim)
    )

    for i in tqdm(range(0, len(urls), batch_size), desc=f"Embedding {file_name}"):
        batch_urls = urls[i:i + batch_size]

        batch_embeddings = bert_model.encode(
            batch_urls,
            batch_size=batch_size,
            show_progress_bar=False,
            device=device
        )

        embeddings[i:i + len(batch_embeddings)] = batch_embeddings.astype("float32")

    embeddings.flush()

    print(file_name, embeddings.shape)
    return embeddings

In [ ]:
df_bert_embeddings = make_bert_embeddings(
    data=df,
    file_name="/kaggle/working/bert_embeddings.dat",
    bert_model=bert_model,
    device=device,
    url_col="url",
    batch_size=32
)
test_bert_embeddings = make_bert_embeddings(
    data=test_all,
    file_name="/kaggle/working/test_bert_embeddings.dat",
    bert_model=bert_model,
    device=device,
    url_col="url",
    batch_size=32
)

In [52]:
embedding_dim = 768

train_bert_embeddings = np.memmap(
    "/kaggle/input/datasets/rasyidbomantoro/phising-urls/bert_embeddings.dat",
    dtype="float32",
    mode="r",
    shape=(len(df), embedding_dim)
)

test_bert_embeddings = np.memmap(
    "/kaggle/input/datasets/rasyidbomantoro/phising-urls/test_bert_embeddings.dat",
    dtype="float32",
    mode="r",
    shape=(len(test_all), embedding_dim) 
)

# Model Initialization

In [40]:
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_output):

        score = self.v(torch.tanh(self.W(lstm_output)))

        weights = torch.softmax(score, dim=1)

        context = torch.sum(weights * lstm_output, dim=1)

        return context
class PhishingBiLSTMAttention(nn.Module):
    def __init__(self, bert_dim=768, manual_dim=20, lstm_hidden=256):
        super().__init__()

        self.bilstm = nn.LSTM(
            input_size=bert_dim,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.attention = AttentionLayer(hidden_dim=lstm_hidden * 2)

        self.manual_dense = nn.Sequential(
            nn.Linear(manual_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        self.classifier = nn.Sequential(
            nn.Linear((lstm_hidden * 2) + 64, 1),
            nn.Sigmoid()
        )

    def forward(self, bert_x, manual_x):
        lstm_out, _ = self.bilstm(bert_x)

        context = self.attention(lstm_out)

        manual_out = self.manual_dense(manual_x)

        combined = torch.cat([context, manual_out], dim=1)

        output = self.classifier(combined)

        return output.squeeze(1)

# Feature Generation 

In [48]:
def make_feature_df(data, url_col="url", label_col="type"):
    rows = []

    for idx, row in tqdm(data.iterrows(), total=len(data)):
        url = row[url_col]
        features = extract_features(url)

        if features is None:
            continue

        features["original_idx"] = idx
        features["url"] = url

        if label_col in data.columns:
            features["type"] = row[label_col]

        rows.append(features)

    features_df = pd.DataFrame(rows)
    return features_df

In [49]:
df_features = make_feature_df(df, url_col="url", label_col="type")
test_features = make_feature_df(test_all, url_col="url", label_col="type")


100%|██████████| 504982/504982 [00:48<00:00, 10332.70it/s]


In [53]:
train_idx_original = df_features["original_idx"].values
test_idx_original = test_features["original_idx"].values

train_bert_embeddings_aligned = train_bert_embeddings[train_idx_original]
test_bert_embeddings_aligned = test_bert_embeddings[test_idx_original]

print("===== TRAIN =====")
print("df_features rows :", len(df_features))
print("train idx rows   :", len(train_idx_original))
print("BERT aligned     :", train_bert_embeddings_aligned.shape)

print("\n===== TEST =====")
print("test_features rows:", len(test_features))
print("test idx rows     :", len(test_idx_original))
print("BERT aligned      :", test_bert_embeddings_aligned.shape)

===== TRAIN =====
df_features rows : 549327
train idx rows   : 549327
BERT aligned     : (549327, 768)

===== TEST =====
test_features rows: 504982
test idx rows     : 504982
BERT aligned      : (504982, 768)


In [54]:
feature_cols = [
    "url_length",
    "number_of_dots_in_url",
    "having_repeated_digits_in_url",
    "number_of_digits_in_url",
    "number_of_special_char_in_url",
    "number_of_hyphens_in_url",
    "number_of_underline_in_url",
    "number_of_slash_in_url",
    "number_of_questionmark_in_url",
    "number_of_equal_in_url",
    "number_of_at_in_url",
    "number_of_dollar_in_url",
    "number_of_exclamation_in_url",
    "number_of_hashtag_in_url",
    "number_of_percent_in_url",
    "path_length",
    "having_query",
    "having_fragment",
    "having_anchor",
    "entropy_of_url"
]

df_features = df_features.dropna(subset=["type"]).reset_index(drop=True)

X_manual = df_features[feature_cols].values.astype("float32")
y = df_features["type"].values.astype("float32")

print(np.unique(y, return_counts=True))

(array([0., 1.], dtype=float32), array([392907, 156420]))


In [56]:
indices = np.arange(len(df_features))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.15,
    random_state=42,
    stratify=y.astype(int)
)

X_bert_train = train_bert_embeddings[train_idx]
X_bert_val = train_bert_embeddings[val_idx]

X_manual_train = X_manual[train_idx]
X_manual_val = X_manual[val_idx]

y_train = y[train_idx]
y_val = y[val_idx]

print("Train:", X_bert_train.shape, X_manual_train.shape, y_train.shape)
print("Val:", X_bert_val.shape, X_manual_val.shape, y_val.shape)

Train: (466927, 768) (466927, 20) (466927,)
Val: (82400, 768) (82400, 20) (82400,)


In [57]:
scaler = StandardScaler()

X_manual_train = scaler.fit_transform(X_manual_train).astype("float32")
X_manual_val = scaler.transform(X_manual_val).astype("float32")

In [61]:
class URLDataset(Dataset):
    def __init__(self, bert_embeddings, manual_features, labels):
        self.bert_embeddings = bert_embeddings
        self.manual_features = manual_features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        bert_x = torch.tensor(self.bert_embeddings[idx], dtype=torch.float32)

        bert_x = bert_x.unsqueeze(0)

        manual_x = torch.tensor(self.manual_features[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)

        return bert_x, manual_x, y

In [66]:
batch_size = 16

train_dataset = URLDataset(X_bert_train, X_manual_train, y_train)
val_dataset = URLDataset(X_bert_val, X_manual_val, y_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [67]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = PhishingBiLSTMAttention(
    bert_dim=768,
    manual_dim=20,
    lstm_hidden=256
).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

Device: cuda


# Training

In [68]:
epochs = 50
patience = 50

best_val_loss = float("inf")
counter = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": []
}

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for bert_x, manual_x, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Train"):
        bert_x = bert_x.to(device, non_blocking=True)
        manual_x = manual_x.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()

        preds = model(bert_x, manual_x)
        loss = criterion(preds, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)


    model.eval()
    val_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for bert_x, manual_x, y_batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - Val"):
            bert_x = bert_x.to(device, non_blocking=True)
            manual_x = manual_x.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            preds = model(bert_x, manual_x)
            loss = criterion(preds, y_batch)

            val_loss += loss.item()

            pred_label = (preds >= 0.5).int().cpu().numpy()
            true_label = y_batch.int().cpu().numpy()

            all_preds.extend(pred_label)
            all_labels.extend(true_label)

    val_loss = val_loss / len(val_loader)

    val_acc = accuracy_score(all_labels, all_preds)
    val_precision = precision_score(all_labels, all_preds, zero_division=0)
    val_recall = recall_score(all_labels, all_preds, zero_division=0)
    val_f1 = f1_score(all_labels, all_preds, zero_division=0)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_precision"].append(val_precision)
    history["val_recall"].append(val_recall)
    history["val_f1"].append(val_f1)

    print(
        f"\nEpoch {epoch+1}/{epochs}"
        f"\nTrain Loss : {train_loss:.4f}"
        f"\nVal Loss   : {val_loss:.4f}"
        f"\nVal Acc    : {val_acc:.4f}"
        f"\nVal Prec   : {val_precision:.4f}"
        f"\nVal Recall : {val_recall:.4f}"
        f"\nVal F1     : {val_f1:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler": scaler,
            "feature_cols": feature_cols,
            "history": history
        }, "/kaggle/working/best_bilstm_attention_model.pt")

        print("Best model saved.")
    else:
        counter += 1
        print(f"No improvement: {counter}/{patience}")

    if counter >= patience:
        print("Early stopping.")
        break

Epoch 1/50 - Val: 100%|██████████| 5150/5150 [00:08<00:00, 641.05it/s]



Epoch 1/50
Train Loss : 0.2000
Val Loss   : 0.1588
Val Acc    : 0.9387
Val Prec   : 0.9349
Val Recall : 0.8434
Val F1     : 0.8868
Best model saved.


Epoch 2/50 - Val: 100%|██████████| 5150/5150 [00:07<00:00, 655.48it/s]



Epoch 2/50
Train Loss : 0.1426
Val Loss   : 0.1336
Val Acc    : 0.9495
Val Prec   : 0.9248
Val Recall : 0.8952
Val F1     : 0.9098
Best model saved.


Epoch 3/50 - Val: 100%|██████████| 5150/5150 [00:07<00:00, 656.36it/s]


...


Epoch 48/50
Train Loss : 0.0246
Val Loss   : 0.0586
Val Acc    : 0.9798
Val Prec   : 0.9647
Val Recall : 0.9643
Val F1     : 0.9645
Best model saved.


Epoch 49/50 - Val: 100%|██████████| 5150/5150 [00:07<00:00, 645.69it/s]



Epoch 49/50
Train Loss : 0.0238
Val Loss   : 0.0616
Val Acc    : 0.9786
Val Prec   : 0.9544
Val Recall : 0.9711
Val F1     : 0.9627
No improvement: 1/50


Epoch 50/50 - Val: 100%|██████████| 5150/5150 [00:07<00:00, 653.61it/s]



Epoch 50/50
Train Loss : 0.0230
Val Loss   : 0.0593
Val Acc    : 0.9797
Val Prec   : 0.9663
Val Recall : 0.9624
Val F1     : 0.9643
No improvement: 2/50


# Test Prediction

In [69]:
checkpoint = torch.load(
    "/kaggle/working/best_bilstm_attention_model.pt",
    map_location=device,
    weights_only=False
)

scaler = checkpoint["scaler"]
feature_cols = checkpoint["feature_cols"]

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

PhishingBiLSTMAttention(
  (bilstm): LSTM(768, 256, batch_first=True, bidirectional=True)
  (attention): AttentionLayer(
    (W): Linear(in_features=512, out_features=512, bias=True)
    (v): Linear(in_features=512, out_features=1, bias=False)
  )
  (manual_dense): Sequential(
    (0): Linear(in_features=20, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=576, out_features=1, bias=True)
    (1): Sigmoid()
  )
)

In [71]:
X_test_manual = test_features[feature_cols].values.astype("float32")
y_test = test_features["type"].values.astype("float32")
X_test_manual = scaler.transform(X_test_manual).astype("float32")


In [72]:
test_bert_embeddings = np.memmap(
    "/kaggle/input/datasets/rasyidbomantoro/phising-urls/test_bert_embeddings.dat",
    dtype="float32",
    mode="r",
    shape=(len(test_features), 768)
)


In [73]:
test_dataset = URLDataset(test_bert_embeddings, X_test_manual, y_test)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [74]:
def evaluate_model(model, data_loader, device, name="Test"):
    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for bert_x, manual_x, y_batch in tqdm(data_loader, desc=f"Evaluating {name}"):
            bert_x = bert_x.to(device, non_blocking=True)
            manual_x = manual_x.to(device, non_blocking=True)

            probs = model(bert_x, manual_x)
            pred_label = (probs >= 0.5).int().cpu().numpy()

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(pred_label)
            all_labels.extend(y_batch.int().numpy())

    print(f"\n===== {name} Result =====")
    print("Accuracy :", accuracy_score(all_labels, all_preds))
    print("Precision:", precision_score(all_labels, all_preds, zero_division=0))
    print("Recall   :", recall_score(all_labels, all_preds, zero_division=0))
    print("F1       :", f1_score(all_labels, all_preds, zero_division=0))

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return np.array(all_probs), np.array(all_preds), np.array(all_labels)

In [75]:
test_probs, test_preds, test_labels = evaluate_model(
    model,
    test_loader,
    device,
    name="Test"
)

Evaluating Test: 100%|██████████| 31562/31562 [01:37<00:00, 324.32it/s]



===== Test Result =====
Accuracy : 0.8772055241572967
Precision: 0.7799337843673528
Recall   : 0.8506128959332848
F1       : 0.8137414806604609

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.89      0.91    345738
           1       0.78      0.85      0.81    159244

    accuracy                           0.88    504982
   macro avg       0.85      0.87      0.86    504982
weighted avg       0.88      0.88      0.88    504982


Confusion Matrix:
[[307518  38220]
 [ 23789 135455]]


In [ ]:
FileLink("best_bilstm_attention_model.pt")

/kaggle/working/best_bilstm_attention_model.pt